# Train eWaSR trên Kaggle với MaSTr1325
Bật **GPU** và gắn dataset MaSTr1325 vào mục **Input** trước khi chạy.

In [ ]:
# Chỉ cần sửa URL này.
REPO_URL = "https://github.com/USERNAME/maritime_kd.git"
EPOCHS = 50
BATCH_SIZE = 4
HEIGHT = 384
WIDTH = 512
MODEL = "ewasr_resnet18"
OUTPUT_NAME = "ewasr_fp32"

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

if "USERNAME" in REPO_URL:
    raise ValueError("Hãy thay USERNAME trong REPO_URL bằng tài khoản GitHub của bạn")

repo_dir = Path("/kaggle/working/maritime_kd")
if repo_dir.exists():
    shutil.rmtree(repo_dir)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Repository:", repo_dir)

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA chưa được bật. Chọn Accelerator > GPU trong Kaggle.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
input_root = Path("/kaggle/input")
candidates = sorted({p.parent for p in input_root.rglob("images") if p.is_dir() and (p.parent / "masks").is_dir()})
if not candidates:
    raise FileNotFoundError("Không tìm thấy thư mục có cả images/ và masks/ trong /kaggle/input")
DATA_ROOT = candidates[0]
print("MaSTr1325:", DATA_ROOT)
print("Images:", len(list((DATA_ROOT / 'images').glob('*'))))
print("Masks:", len(list((DATA_ROOT / 'masks').glob('*'))))

In [ ]:
split_dir = Path("/kaggle/working/splits")
subprocess.run([
    sys.executable, "prepare_splits.py",
    "--data-root", str(DATA_ROOT),
    "--output-dir", str(split_dir),
    "--seed", "42",
], check=True)

In [ ]:
output_dir = Path("/kaggle/working/outputs") / OUTPUT_NAME
command = [
    sys.executable, "train_student.py",
    "--data-root", str(DATA_ROOT),
    "--split-dir", str(split_dir),
    "--model", MODEL,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--height", str(HEIGHT),
    "--width", str(WIDTH),
    "--device", "cuda",
    "--amp",
    "--output-dir", str(output_dir),
]
print("Running:", " ".join(command))
subprocess.run(command, check=True)

In [ ]:
subprocess.run([
    sys.executable, "evaluate.py",
    "--checkpoint", str(output_dir / "best.pt"),
    "--data-root", str(DATA_ROOT),
    "--split-dir", str(split_dir),
    "--split", "test",
    "--batch-size", str(BATCH_SIZE),
    "--device", "cuda",
], check=True)
print("Kết quả được lưu tại:", output_dir)